In [67]:
# A generator main graph and a translator subgraph are created. The main graph generates paragraphs on a given topic and then translates them into a specified language using the subgraph.

In [52]:
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START,END
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import os
load_dotenv()

True

In [53]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

MAIN_LLM=ChatOpenAI (
    model= 'gpt-4o-mini',
    temperature=0.3,
    api_key=OPENAI_API_KEY,
    max_retries=3
)

FALLBACK_LLM=ChatGoogleGenerativeAI(
    model= 'gemini-1.5-turbo',
    temperature=0.3,
    api_key=GEMINI_API_KEY
)

llm=MAIN_LLM.with_fallbacks([FALLBACK_LLM])

In [54]:
''' 
PS: we are making a mulitagent system, where generate agent generates the paragraphs on some topic given by user
and then the translator agent, translates the paragraphs into some other language given by the user.

our plan: one main graph, and one subgraph , we will make subgraph for the translate agent.
'''

' \nPS: we are making a mulitagent system, where generate agent generates the paragraphs on some topic given by user\nand then the translator agent, translates the paragraphs into some other language given by the user.\n\nour plan: one main graph, and one subgraph , we will make subgraph for the translate agent.\n'

In [55]:
#subgraph 
class TranslateInput(BaseModel):
    sentence :str=Field(..., description="The sentence to be translated")
    lang: str=Field(description="The language to translate the sentence into",default="hindi")
    output:str=Field(description="The translated sentence",default=None)

In [56]:
from langchain_core.prompts import ChatPromptTemplate
def TranslatorNode(State: TranslateInput):
    prompt=ChatPromptTemplate.from_template(
    f'''
    You are a Translator agent. Your only job is to translate the `input` to the `lang` language.
    `input` {State.sentence}
    `lang` {State.lang}
''')

    chain=prompt | llm
    res=chain.invoke({"input":State.sentence, "lang":State.lang})

    return {"output":res.content}

In [57]:
builder=StateGraph(TranslateInput)

builder.add_node("TranslatorNode", TranslatorNode)

builder.add_edge(START, "TranslatorNode")
builder.add_edge("TranslatorNode", END)

In [58]:
translater_subgraph=builder.compile()

In [59]:
# res=translater_subgraph.invoke({"sentence":"Hello, you are very nice","lang":"French"})
# print(res)

In [60]:
class mainGraph(BaseModel):
    topic:str=Field(..., description="The topic to generate paragraphs on")
    lang: str=Field(..., description="The language to translate the paragraphs into")
    Translated_into: str=Field(description="The language into which the paragraphs are translated",default="Hindi")
    generated_text:str=Field(description="The translated paragraphs",default=None)
    final_output:str=Field(description="The final output after translation",default=None)

In [61]:
def GeneraterNode(State:mainGraph):

    prompt=ChatPromptTemplate.from_template(
    f'''
you are a content generator agent. Your job is to write 100 char paragraphs on the given topic only in given language.
 The topic is {State.topic}.
 Generate into {State.lang} Language.
''')
    
    chain=prompt | llm
    res=chain.invoke({"topic":State.topic,"lang":State.lang})

    return {"generated_text":res.content}

In [62]:
def translaterNode(State:mainGraph):
    res=translater_subgraph.invoke({"sentence":State.generated_text,"lang":State.Translated_into})
    return {
        "users query":State.topic,
        "Generated text:":State.generated_text,
        "final_output":res["output"]}

In [63]:
MainGraphBuilder=StateGraph(mainGraph)
MainGraphBuilder.add_node("GeneraterNode", GeneraterNode)
MainGraphBuilder.add_node("translaterNode", translaterNode)

MainGraphBuilder.add_edge(START, "GeneraterNode")
MainGraphBuilder.add_edge("GeneraterNode", "translaterNode")
MainGraphBuilder.add_edge("translaterNode", END)

In [66]:
MainGraph=MainGraphBuilder.compile()
res=MainGraph.invoke({"topic":"Friendship","lang":"english","Translated_into":"spanish"})
import pprint
pprint.pprint(res)

{'Translated_into': 'spanish',
 'final_output': 'La amistad es un vínculo que trasciende el tiempo y la '
                 'distancia, ofreciendo apoyo, alegría y comprensión. Los '
                 'verdaderos amigos se levantan mutuamente, comparten risas y '
                 'brindan consuelo en momentos difíciles. Esta conexión '
                 'fomenta la confianza y la lealtad, creando recuerdos que '
                 'duran toda la vida. En momentos de celebración o tristeza, '
                 'los amigos están a tu lado, haciendo que el viaje de la vida '
                 'sea más significativo. Nutrir estas relaciones requiere '
                 'esfuerzo y comunicación, pero las recompensas son '
                 'incalculables. Un verdadero amigo te ve por quien eres, '
                 'aceptando tus defectos y celebrando tus fortalezas. Abraza '
                 'la belleza de la amistad; enriquece nuestras vidas de '
                 'innumerables maneras.',
 'generate